# Student Enrollment Funnel & A/B Testing

Executed synthetic-data case study. No real students or campaigns. The Tableau workbook is pending; exports and dashboard instructions are complete.

This notebook executes the source pipeline and shows actual computed outputs. The seed and assumptions are fixed before generation. See DATA_DICTIONARY.md for the encoded treatment effect.


In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'run_analysis.py').exists():
    ROOT = ROOT / 'projects' / 'student-enrollment-funnel'
if not (ROOT / 'run_analysis.py').exists():
    raise RuntimeError('Run the notebook from the project folder or repository root.')
sys.path.insert(0, str(ROOT.resolve()))
import run_analysis as analysis
print(json.dumps(analysis.EXPERIMENT, indent=2))


{
  "unit": "unique synthetic prospect",
  "allocation": "independent 50/50 Bernoulli",
  "primary_outcome": "inquiry within 7 days of first visit",
  "control": "A: standard form",
  "treatment": "B: simplified form",
  "sample_size": 10000,
  "alpha": 0.05,
  "srm_threshold": 0.01,
  "test": "two-sided pooled two-proportion z test",
  "difference_ci": "95% Newcombe interval from Wilson score bounds",
  "analysis": "intention-to-treat, fixed sample; no peeking or subgroup significance claims",
  "cohort_start": "2025-01-01",
  "cohort_end": "2025-06-30",
  "followup_days": 90,
  "data_cutoff": "2025-09-28",
  "planning_assumptions": "20% baseline; 3 percentage-point absolute MDE; 80% power"
}


## Generate and validate

One row per prospect; independent assignment; nested stages. Every cohort has 90 days of observation. IDs are fabricated.


In [2]:
p, costs = analysis.generate()
analysis.validate(p, costs)
print(f'Validated {len(p):,} prospects and {len(costs):,} daily channel costs.')
print(p[['prospect_id','channel','device','program','variant','inquiry','enrolled']].head().to_string(index=False))


Validated 10,000 prospects and 905 daily channel costs.
prospect_id     channel  device      program variant  inquiry  enrolled
     P00001 Paid Social Desktop Data Science       A        0         0
     P00002 Paid Search  Mobile Data Science       B        0         0
     P00003       Email Desktop   Healthcare       A        0         0
     P00004 Paid Search Desktop     Business       A        1         0
     P00005 Paid Social  Mobile   Healthcare       A        1         0


## SQL funnel and cost analysis

Read actual queries from analysis.sql. Aggregate audience and costs separately before joining. Reconcile outputs against row-level counts.


In [3]:
tables = analysis.sql_exports(p, costs)
m = tables['monthly_channel']
print(m.head().to_string(index=False))
print('Total spend:', round(m.spend.sum(), 2))
print('Stage counts:')
print(m[analysis.STAGES].sum().to_string())


cohort_month        channel  visitors  inquiries  applications  admissions  enrollments   spend  inquiry_rate  enrollment_rate  cost_per_inquiry  cost_per_enrollment
     2025-01          Email       237         72            40          27           19 1445.06      0.303797         0.080169         20.070278            76.055789
     2025-01 Organic Search       342         86            54          41           27 2477.16      0.251462         0.078947         28.804186            91.746667
     2025-01    Paid Search       475         98            52          34           14 7029.47      0.206316         0.029474         71.729286           502.105000
     2025-01    Paid Social       444         78            36          28           11 4593.51      0.175676         0.024775         58.891154           417.591818
     2025-01       Referral       172         36            20          15            8  903.70      0.209302         0.046512         25.102778           112.962500
Tota

## Simulated randomized experiment

Intention-to-treat comparison of seven-day inquiry conversion, two-sided alpha=.05. A fixed 3-point uplift is encoded in the generator. The resulting estimate illustrates inference inside a simulation.


In [4]:
ab = analysis.experiment(p)
print(json.dumps(ab, indent=2))


{
  "control_n": 4979,
  "treatment_n": 5021,
  "control_inquiries": 959,
  "treatment_inquiries": 1141,
  "control_rate": 0.19260895762201244,
  "treatment_rate": 0.22724556861183032,
  "absolute_lift_pp": 3.4636610989817878,
  "relative_lift_pct": 17.982866122867907,
  "ci_low_pp": 1.8677438888613063,
  "ci_high_pp": 5.057333870009903,
  "z_statistic": 4.251854288678517,
  "p_value": 2.1200778886751595e-05,
  "srm_p_value": 0.6744854536964989,
  "planned_n_per_arm": 2943,
  "decision": "positive simulated effect"
}


## Regression without outcome leakage

Use features available at first visit. Evaluate on a disjoint stratified holdout. Compare calibration losses to a constant-probability baseline. Regularized coefficients are associations, not causal effects.


In [5]:
model = analysis.regression(p)
print(json.dumps(model, indent=2))


{
  "train_n": 7500,
  "test_n": 2500,
  "test_enrollments": 114,
  "test_prevalence": 0.0456,
  "roc_auc": 0.6313804208761635,
  "average_precision": 0.07876984534865071,
  "brier_score": 0.0430936710726095,
  "baseline_brier": 0.04352071111111112,
  "log_loss": 0.18070296844216654,
  "baseline_log_loss": 0.18535081526061895,
  "reference_categories": {
    "channel": "Email",
    "device": "Desktop",
    "program": "Business",
    "variant": "A"
  }
}


## Results and limitations

![Overview](results/overview.png)

The simulated inquiry lift is approximately 3.46 percentage points. This is not a real-world business gain. The regression has modest predictive ability. See results/business_brief.md for recommendations and limitations, and TABLEAU_GUIDE.md for the remaining native Tableau authoring steps.
